In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean

In [ ]:
# 2: Load Dataset (Wang Corel-1K)
# Folder structure: dataset/0/, dataset/1/, ..., dataset/9/
# 0=Africans  1=Beach  2=Buildings  3=Buses  4=Dinosaurs
# 5=Elephants  6=Flowers  7=Horses  8=Mountains  9=Food

dataset_path = " "   # <-- UPDATE THIS PATH

images, labels, paths = [], [], []

for category in range(10):
    folder = os.path.join(dataset_path, str(category))
    for img_file in os.listdir(folder):
        img_path = os.path.join(folder, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (128, 128))
        images.append(img)
        labels.append(category)
        paths.append(img_path)

images = np.array(images)
labels = np.array(labels)

print(f"Loaded {len(images)} images from Wang Corel-1K dataset")
print(f"   Categories found: {np.unique(labels)}")

FileNotFoundError: [Errno 2] No such file or directory: ' /0'

In [ ]:
#3: Preprocessing with CLAHE

def preprocess_image(img):
    """
    Applies CLAHE on the L channel (LAB color space)
    to normalize contrast before feature extraction.
    """
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)

    merged_lab = cv2.merge((cl, a_channel, b_channel))
    enhanced_img = cv2.cvtColor(merged_lab, cv2.COLOR_LAB2BGR)

    return enhanced_img

# Quick test
sample = preprocess_image(images[0])
print(f"CLAHE preprocessing OK. Output shape: {sample.shape}")

IndexError: list index out of range

In [ ]:

#  4: Build Feature Index for All Dataset Images

print("Building feature index... please wait ⏳")

feature_index = []
for i, img in enumerate(images):
    feat = get_fused_features(img)
    feature_index.append(feat)
    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(images)} images...")

feature_index = np.array(feature_index)

# Save so you don't recompute every time
np.save('feature_index.npy', feature_index)
np.save('labels.npy', labels)

print(f"\n Feature index shape: {feature_index.shape}")
print(f" Saved to feature_index.npy and labels.npy")

Building feature index... please wait ⏳

 Feature index shape: (0,)
 Saved to feature_index.npy and labels.npy


In [ ]:

#  4: Color Histogram Feature Extraction

def extract_color_features(img):
    """
    Extracts a 96-dimensional HSV color histogram.
    32 bins x 3 channels = 96 features.
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hist_features = []

    for ch in range(3):
        h = cv2.calcHist([hsv], [ch], None, [32], [0, 256])
        cv2.normalize(h, h)
        hist_features.extend(h.flatten())

    return np.array(hist_features)  # shape: (96,)

# Quick test
color_vec = extract_color_features(images[0])
print(f" Color feature vector size: {color_vec.shape}")

In [ ]:

# 5: Edge Feature Extraction (Canny + Sobel)

def extract_edge_features(img):
    """
    Uses Canny edge detection + Sobel gradients to extract:
    - Edge density (1 value)
    - Orientation histogram (8 bins)
    Total: 9 features
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Canny edge density
    edges = cv2.Canny(gray, threshold1=100, threshold2=200)
    edge_density = np.sum(edges > 0) / edges.size

    # Gradient orientation histogram
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    angles = np.arctan2(sobely, sobelx) * (180 / np.pi)
    angles[angles < 0] += 360

    hist, _ = np.histogram(angles, bins=8, range=(0, 360))
    hist = hist / (hist.sum() + 1e-10)  # normalize

    edge_vec = np.concatenate([[edge_density], hist])
    return edge_vec  # shape: (9,)

# Quick test
edge_vec = extract_edge_features(images[0])
print(f"" Edge feature vector size: {edge_vec.shape}")

In [ ]:

#  6: Feature Fusion (Color 70% + Edge 30%)

def get_fused_features(img):
    """
    Preprocesses image, extracts Color and Edge features,
    normalizes each, then fuses with weights:
    70% Color + 30% Edge
    """
    enhanced = preprocess_image(img)

    color_vec = extract_color_features(enhanced)
    edge_vec  = extract_edge_features(enhanced)

    # L2 normalize
    color_vec = color_vec / (np.linalg.norm(color_vec) + 1e-10)
    edge_vec  = edge_vec  / (np.linalg.norm(edge_vec)  + 1e-10)

    # Weighted concatenation
    fused = np.concatenate([
        color_vec * 0.70,
        edge_vec  * 0.30
    ])

    return fused  # shape: (105,)

# Quick test
fused = get_fused_features(images[0])
print(f" Fused feature vector size: {fused.shape}")

In [ ]:

#  8: Query & Retrieve Top-5 (Cosine + Euclidean)


def retrieve_top5_cosine(query_img, feature_index):
    """Retrieves top-5 using Cosine Similarity (higher = more similar)."""
    query_feat = get_fused_features(query_img)
    scores = cosine_similarity([query_feat], feature_index)[0]
    top5 = np.argsort(scores)[::-1][:5]
    return top5, scores

def retrieve_top5_euclidean(query_img, feature_index):
    """Retrieves top-5 using Euclidean Distance (lower = more similar)."""
    query_feat = get_fused_features(query_img)
    distances = np.array([euclidean(query_feat, f) for f in feature_index])
    scores = 1 / (1 + distances)   # convert to similarity score
    top5 = np.argsort(distances)[:5]
    return top5, scores


# ----- Run a Query -----
query_idx = 0             # change this to test different images
query_img   = images[query_idx]
query_label = labels[query_idx]

top5_cos, scores_cos = retrieve_top5_cosine(query_img, feature_index)
top5_euc, scores_euc = retrieve_top5_euclidean(query_img, feature_index)

print(f"Query category : {query_label}")
print(f"\nCosine  Top-5 indices : {top5_cos}")
print(f"Cosine  Top-5 scores  : {[round(scores_cos[i], 4) for i in top5_cos]}")
print(f"\nEuclidean Top-5 indices : {top5_euc}")
print(f"Euclidean Top-5 scores  : {[round(scores_euc[i], 4) for i in top5_euc]}")